In [5]:
#%pip install msoffcrypto-tool
#%pip install pywin32

In [6]:
#%pip install python-calamine

In [7]:
import msoffcrypto
import pandas as pd
import io
import openpyxl

In [8]:
pd.set_option('display.max_rows', None)

In [9]:
# 1. กำหนดชื่อไฟล์และรหัสผ่าน
file_path = r'H:\My Drive\การเงิน\ยอดขาย\2569\ยอดขายรวมทุกสาขาBplus2569.xlsx'
password = '18651865'

In [10]:
data = []
month_names = ['ม.ค.69', 'ก.พ.69', 'มี.ค.69', 'เม.ย.69', 'พ.ค.69', 'มิ.ย.69', 
               'ก.ค.69', 'ส.ค.69', 'ก.ย.69', 'ต.ค.69', 'พ.ย.69', 'ธ.ค.69']
# 2. สร้างออบเจ็กต์สำหรับจัดการไฟล์ที่ล็อก
temp_file = io.BytesIO()    

with open(file_path, 'rb') as f:
    office_file = msoffcrypto.OfficeFile(f)
    
    # ใส่รหัสผ่านเพื่อปลดล็อก
    office_file.load_key(password=password)
    
    # บันทึกไฟล์ที่ปลดล็อกแล้วลงในหน่วยความจำ (temp_file)
    office_file.decrypt(temp_file)

In [11]:
cols = 'A:H,T'
# 3. ใช้ Pandas อ่านไฟล์จากหน่วยความจำ
for i in month_names:
    df = pd.read_excel(temp_file , sheet_name= i ,usecols=cols, header= 4 ,engine= 'openpyxl')
    df = df.iloc[0:35,:]
    # แปลงข้อมูลเป็นวันที่
    df['DATE'] = pd.to_datetime(df['Unnamed: 0'], dayfirst=True ,errors='coerce')
    #ลบค่าว่าง
    df.dropna(subset='DATE',inplace=True)
    #กรอกวันที่ที่มากกว่า 2020
    df = df[df['DATE'].dt.year > 2020]

    df['DATE'] = df['DATE'].dt.date

    df.drop(columns=['Unnamed: 0'] , inplace=True)
    
    df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

    data.append(df)

full = pd.concat(data , ignore_index=True)
# ลบซ้ำ
full.drop_duplicates(inplace=True)

full.insert(0 ,'DATE1',full['DATE'])

full.drop(columns='DATE',inplace=True)

full.rename(columns={'DATE1':'DATE'},inplace=True)



In [12]:
full

,DATE,ยอดขายเค1,ยอดขายเค2,ยอดขายเค3,ยอดขายเค4,ยอดขายเค5,ยอดขายSP,ยอดขายPet,WH
0,2026-01-01,108987,45336,69713,26216,31895,155215,14268,1190971.32
1,2026-01-02,88718,45500,60439,24561,29012,160030,18760,1131370.25
2,2026-01-03,84000,36760,51683,23059,30164,117272,20929,1194728.22
3,2026-01-04,71691,35618,48282,20816,27359,128854,10512,1041403.71
4,2026-01-05,73485,34361,50409,17948,23496,140967,18533,1173480.63
5,2026-01-06,68945,33590,45253,22376,24339,135655,16162,819157.04
6,2026-01-07,62913,34560,45480,21405,24168,131370,16950,1062041.64
7,2026-01-08,59935,37379,42492,18569,22511,103381,13654,1138786.03
8,2026-01-09,66614,37396,52310,21416,23625,103202,22447,2003329.73
9,2026-01-10,61997,39174,59803,20716,20916,100977,16234,1004557.17
